# Simulación del Mundial 2026

En este notebook se simulará el Mundial 2026 utilizando el modelo de Machine Learning seleccionado previamente.

El objetivo es:

- Simular la fase de grupos.
- Calcular la clasificación de cada grupo.
- Determinar los equipos clasificados.
- Simular las rondas eliminatorias.
- Obtener un campeón del torneo.

In [1]:
import pandas as pd
import numpy as np

In [2]:
selecciones = pd.read_csv(
    "../data/processed/selecciones_mundial_2026.csv"

)

dataset_final = pd.read_csv(
    "../data/processed/dataset_mundial_final.csv"
)

In [3]:
dataset_final.head(200)

,date,home_team,away_team,home_score,away_score,tournament,tournament_weight,neutral,goal_diff,fifa_rank_home,...,home_last5_goals_against,home_last5_goal_balance,away_last5_points,away_last5_goals_for,away_last5_goals_against,away_last5_goal_balance,form_points_diff,form_goals_for_diff,form_goals_against_diff,form_goal_balance_diff
0,2000-03-19,Cayman Islands,Cuba,0.0,0.0,FIFA World Cup qualification,6,False,0.0,149.0,...,1.0,-1.0,5.0,8.0,7.0,1.0,-4.0,-8.0,-6.0,-2.0
1,2000-03-19,Dominica,Haiti,1.0,3.0,FIFA World Cup qualification,6,False,-2.0,152.0,...,0.0,2.0,7.0,5.0,12.0,-7.0,-4.0,-3.0,-12.0,9.0
2,2000-03-19,Bermuda,British Virgin Islands,9.0,0.0,FIFA World Cup qualification,6,False,9.0,157.0,...,2.0,-2.0,3.0,4.0,8.0,-4.0,-3.0,-4.0,-6.0,2.0
3,2000-03-25,Bahrain,Jordan,1.0,1.0,Friendly,1,False,0.0,139.0,...,1.0,0.0,8.0,6.0,5.0,1.0,-5.0,-5.0,-4.0,-1.0
4,2000-03-25,Zambia,Botswana,3.0,0.0,COSAFA Cup,3,False,3.0,38.0,...,5.0,1.0,4.0,4.0,8.0,-4.0,2.0,2.0,-3.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2001-05-03,Turkmenistan,Uzbekistan,2.0,5.0,FIFA World Cup qualification,6,True,-3.0,131.0,...,0.0,7.0,3.0,2.0,12.0,-10.0,3.0,5.0,-12.0,17.0
196,2001-05-04,Algeria,Morocco,1.0,2.0,FIFA World Cup qualification,6,False,-1.0,76.0,...,7.0,2.0,6.0,2.0,3.0,-1.0,4.0,7.0,4.0,3.0
197,2001-05-05,Sierra Leone,Ghana,1.0,1.0,FIFA World Cup qualification,6,False,0.0,135.0,...,2.0,5.0,4.0,5.0,8.0,-3.0,5.0,2.0,-6.0,8.0
198,2001-05-05,South Africa,Zimbabwe,2.0,1.0,FIFA World Cup qualification,6,False,1.0,22.0,...,2.0,4.0,9.0,7.0,6.0,1.0,1.0,-1.0,-4.0,3.0


In [4]:
selecciones.head()

,team,rank,total_points,elo
0,Mexico,15.0,1675.75,1835.0
1,South Africa,61.0,1426.73,1531.0
2,South Korea,22.0,1599.45,1784.0
3,Czech Republic,44.0,1487.00,1731.0
4,Canada,27.0,1559.15,1802.0


In [5]:
selecciones.shape

(48, 4)

In [6]:
grupos = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],

    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],

    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],

    "D": ["United States", "Paraguay", "Australia", "Turkey"],

    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],

    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],

    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],

    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],

    "I": ["France", "Senegal", "Iraq", "Norway"],

    "J": ["Argentina", "Algeria", "Austria", "Jordan"],

    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],

    "L": ["England", "Croatia", "Ghana", "Panama"]
}

In [7]:
total_equipos = 0

for grupo in grupos.values():

    total_equipos += len(grupo)

print(total_equipos)

48


In [8]:
partidos_grupos = []

In [9]:
for nombre_grupo, equipos in grupos.items():

    for i in range(len(equipos)):

        for j in range(i + 1, len(equipos)):

            partidos_grupos.append(
                [
                    nombre_grupo,
                    equipos[i],
                    equipos[j]
                ]
            )

In [10]:
partidos_grupos = pd.DataFrame(
    partidos_grupos,
    columns=[
        "grupo",
        "home_team",
        "away_team"
    ]
)

In [11]:
partidos_grupos.head(10)

,grupo,home_team,away_team
0,A,Mexico,South Africa
1,A,Mexico,South Korea
2,A,Mexico,Czech Republic
3,A,South Africa,South Korea
4,A,South Africa,Czech Republic
5,A,South Korea,Czech Republic
6,B,Canada,Bosnia and Herzegovina
7,B,Canada,Qatar
8,B,Canada,Switzerland
9,B,Bosnia and Herzegovina,Qatar


In [12]:
partidos_grupos.shape

(72, 3)

In [13]:
dataset_final.columns.tolist()

['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'tournament_weight',
 'neutral',
 'goal_diff',
 'fifa_rank_home',
 'fifa_points_home',
 'fifa_rank_away',
 'fifa_points_away',
 'elo_home',
 'elo_away',
 'fifa_rank_diff',
 'fifa_points_diff',
 'elo_diff',
 'home_last5_points',
 'home_last5_goals_for',
 'home_last5_goals_against',
 'home_last5_goal_balance',
 'away_last5_points',
 'away_last5_goals_for',
 'away_last5_goals_against',
 'away_last5_goal_balance',
 'form_points_diff',
 'form_goals_for_diff',
 'form_goals_against_diff',
 'form_goal_balance_diff']

In [14]:
form_home = dataset_final[
    [
        "date",
        "home_team",
        "home_last5_points",
        "home_last5_goals_for",
        "home_last5_goals_against",
        "home_last5_goal_balance"
    ]
].copy()

form_home.columns = [
    "date",
    "team",
    "last5_points",
    "last5_goals_for",
    "last5_goals_against",
    "last5_goal_balance"
]

In [15]:
form_away = dataset_final[
    [
        "date",
        "away_team",
        "away_last5_points",
        "away_last5_goals_for",
        "away_last5_goals_against",
        "away_last5_goal_balance"
    ]
].copy()

form_away.columns = [
    "date",
    "team",
    "last5_points",
    "last5_goals_for",
    "last5_goals_against",
    "last5_goal_balance"
]

In [16]:
form_total = pd.concat(
    [form_home, form_away],
    ignore_index=True
)

In [17]:
form_total.head()

,date,team,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,2000-03-19,Cayman Islands,1.0,0.0,1.0,-1.0
1,2000-03-19,Dominica,3.0,2.0,0.0,2.0
2,2000-03-19,Bermuda,0.0,0.0,2.0,-2.0
3,2000-03-25,Bahrain,3.0,1.0,1.0,0.0
4,2000-03-25,Zambia,6.0,6.0,5.0,1.0


In [18]:
form_total.shape

(36620, 6)

In [19]:
form_total["date"] = pd.to_datetime(
    form_total["date"]
)

form_total = form_total.sort_values(
    "date"
)

In [20]:
form_actual = (
    form_total
    .groupby("team")
    .tail(1)
)

In [48]:
form_actual["team"] = form_actual["team"].replace({
    "Curaçao": "Curacao",
    "Czechia": "Czech Republic",
    "Congo": "DR Congo"

})

In [49]:
form_actual.head()

,date,team,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
31400,2019-09-10,Eritrea,0.0,4.0,13.0,-9.0
16252,2023-10-17,Guam,3.0,6.0,9.0,-3.0
16841,2024-06-11,Mongolia,3.0,3.0,5.0,-2.0
35366,2024-09-09,Tonga,6.0,9.0,20.0,-11.0
35551,2024-10-15,Turks and Caicos Islands,4.0,5.0,10.0,-5.0


In [50]:
len(form_actual)

192

In [51]:
print("Equipos en form_actual:", len(form_actual["team"].unique()))
print("Equipos en mundial_final:", len(dataset_final["home_team"].unique()))

Equipos en form_actual: 192
Equipos en mundial_final: 192


In [53]:
set()

set()

In [54]:
selecciones_actuales = selecciones.merge(
    form_actual,
    on="team",
    how="left"
)

In [55]:
selecciones_actuales.shape

(48, 9)

In [56]:
selecciones_actuales.isna().sum()

team                   0
rank                   0
total_points           0
elo                    0
date                   0
last5_points           0
last5_goals_for        0
last5_goals_against    0
last5_goal_balance     0
dtype: int64

In [60]:
form_actual.shape

(192, 6)

In [61]:
len(form_actual["team"].unique())

192

In [62]:
faltan_form = (
    set(selecciones["team"])
    - set(form_actual["team"])
)

print(sorted(faltan_form))

[]


In [63]:
form_actual.sample(10)

,date,team,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
36553,2025-11-26,Djibouti,1.0,3.0,14.0,-11.0
18179,2025-11-17,Netherlands,11.0,19.0,3.0,16.0
18303,2025-12-30,Uganda,10.0,9.0,5.0,4.0
18302,2025-12-29,Zimbabwe,5.0,6.0,7.0,-1.0
18207,2025-11-18,Guatemala,6.0,8.0,7.0,1.0
36401,2025-10-14,Burundi,6.0,7.0,10.0,-3.0
18148,2025-11-15,Barbados,4.0,9.0,14.0,-5.0
36616,2025-12-31,Burkina Faso,12.0,12.0,1.0,11.0
17651,2025-06-05,Cambodia,8.0,10.0,9.0,1.0
18190,2025-11-18,Trinidad and Tobago,6.0,6.0,6.0,0.0


In [64]:
form_actual.sort_values("date").tail(20)

,date,team,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
18302,2025-12-29,Zimbabwe,5.0,6.0,7.0,-1.0
18301,2025-12-29,Angola,7.0,7.0,7.0,0.0
36609,2025-12-29,Zambia,4.0,5.0,12.0,-7.0
36610,2025-12-29,Mali,5.0,3.0,3.0,0.0
36612,2025-12-29,South Africa,10.0,8.0,4.0,4.0
36611,2025-12-29,Egypt,6.0,5.0,4.0,1.0
36613,2025-12-30,Nigeria,9.0,8.0,5.0,3.0
36615,2025-12-30,Senegal,12.0,19.0,5.0,14.0
18305,2025-12-30,Benin,9.0,6.0,5.0,1.0
18304,2025-12-30,Tanzania,3.0,1.0,6.0,-5.0


In [39]:
form_total.shape

(36620, 6)

In [68]:
dataset_final["date"].max()

'2025-12-31'

In [73]:
dataset_final[["home_team", "away_team"]].head()

,home_team,away_team
0,Cayman Islands,Cuba
1,Dominica,Haiti
2,Bermuda,British Virgin Islands
3,Bahrain,Jordan
4,Zambia,Botswana


In [45]:
dataset_final["home_team"].nunique()

192

In [53]:
dataset_final["away_team"].nunique()

192

In [54]:
dataset_final["date"].min(), dataset_final["date"].max()

('2000-03-19', '2025-12-31')

In [75]:
selecciones_actuales[
    ["team", "rank", "total_points", "elo"]
].head(20)

,team,rank,total_points,elo
0,Mexico,15.0,1675.75,1835.0
1,South Africa,61.0,1426.73,1531.0
2,South Korea,22.0,1599.45,1784.0
3,Czech Republic,44.0,1487.00,1731.0
4,Canada,27.0,1559.15,1802.0
5,Bosnia and Herzegovina,71.0,1362.37,1571.0
6,Qatar,54.0,1454.96,1427.0
7,Switzerland,17.0,1654.69,1897.0
8,Brazil,5.0,1760.46,1979.0
9,Morocco,11.0,1716.34,1830.0


In [77]:
selecciones_actuales.isna().sum()

team                   0
rank                   0
total_points           0
elo                    0
date                   0
last5_points           0
last5_goals_for        0
last5_goals_against    0
last5_goal_balance     0
dtype: int64

In [78]:
selecciones_actuales["last5_points"] = (
    selecciones_actuales["last5_points"]
    .fillna(
        selecciones_actuales["last5_points"].mean()
    )
)

selecciones_actuales["last5_goals_for"] = (
    selecciones_actuales["last5_goals_for"]
    .fillna(
        selecciones_actuales["last5_goals_for"].mean()
    )
)

selecciones_actuales["last5_goals_against"] = (
    selecciones_actuales["last5_goals_against"]
    .fillna(
        selecciones_actuales["last5_goals_against"].mean()
    )
)

selecciones_actuales["last5_goal_balance"] = (
    selecciones_actuales["last5_goal_balance"]
    .fillna(
        selecciones_actuales["last5_goal_balance"].mean()
    )
)

In [80]:
selecciones_actuales.isna().sum()

team                   0
rank                   0
total_points           0
elo                    0
date                   0
last5_points           0
last5_goals_for        0
last5_goals_against    0
last5_goal_balance     0
dtype: int64

In [85]:
selecciones_actuales.to_csv(
    "selecciones_actuales_mundial.csv",
    index=False
)

In [82]:
selecciones_actuales.shape

(48, 9)

In [83]:
faltan_form = (
    set(selecciones["team"])
    - set(form_actual["team"])
)

print(sorted(faltan_form))

[]


In [84]:
print(len(set(selecciones["team"])))
print(len(set(form_actual["team"])))

48
192
